Understanding Vector Stores with ChromaDB in LangChain
This notebook demonstrates how to use ChromaDB as a vector store in LangChain for efficient semantic search and retrieval. Vector stores are essential components in modern RAG (Retrieval Augmented Generation) systems.

What is a Vector Store?
A vector store is a specialized database that stores and retrieves text embeddings (high-dimensional vector representations of text). It enables:

Semantic search (finding similar content)
Efficient retrieval of relevant information
Scalable document storage and querying
Why ChromaDB?
ChromaDB is a popular choice because it offers:

Easy setup and usage
Good performance
Local storage option
Integration with many embedding models
Support for metadata filtering
Notebook Structure
Setup and Configuration
Document Creation
Embedding Generation
Vector Store Operations
Similarity Search
Advanced Queries

# 1 Setup and configuration

In [1]:
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)
logging.getLogger('chromadb').setLevel(logging.ERROR)

In [9]:
from dotenv import load_dotenv
load_dotenv()
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma


# 2 Vectore Store Components
For our vector store setup, we need:

1. Embeddings Model: GoogleGenerativeAIEmbeddings

Converts text into numerical vectors
Uses Google's Gemini model for high-quality embeddings
2. Vector Store: Chroma

Stores and manages the vectors
Provides similarity search capabilities
Persists data locally


In [10]:
try:
    import chromadb
    print(f"ChromaDB version: {chromadb.__version__} ✅")
except ImportError as e:
    print("ChromaDB is not installed properly ❌")

ChromaDB version: 1.5.5 ✅


In [11]:
from langchain_core.documents import Document
doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

# 3 Document Creation and Storage
Here we:

Create Document objects that contain:

page_content: The actual text content
metadata: Additional information about the document
Use sports data as an example:

IPL (Indian Premier League) player information
Team affiliations as metadata
Performance statistics
This structure allows us to:

Store and retrieve content semantically
Filter results based on metadata
Maintain context with the source information

In [12]:

print(type(doc1))

<class 'langchain_core.documents.base.Document'>


In [13]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [14]:
#Initialize Chroma with proper settings
try:
    embedding_function = GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        task_type="retrieval_query"  # Specify the task type
    )
    
    vector_store = Chroma(
        embedding_function=embedding_function,
        persist_directory='my_chroma_db',
        collection_name='sample',
        collection_metadata={"hnsw:space": "cosine"}  # Specify distance metric
    )
    print("Vector store initialized successfully ✅")
except Exception as e:
    print(f"Error initializing vector store: {str(e)} ❌")

Vector store initialized successfully ✅


In [16]:

# Add documents with error handling
try:
    vector_store.add_documents(docs)
    print(f"Successfully added {len(docs)} documents to the vector store ✅")
except Exception as e:
    print(f"Error adding documents: {str(e)} ❌")

Successfully added 5 documents to the vector store ✅


In [17]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['8e8317a5-8519-4222-97e3-10b7a1152b40',
  'f9fbe472-3b41-4c1a-a73c-bbbdab36571e',
  '029250b9-2684-4522-b414-94a3adb95a88',
  '5773d233-319c-43d5-bb57-b9a0e0d2eb18',
  '7582f4d6-fac4-49f1-9379-13b0b5aa51be'],
 'embeddings': array([[ 0.00187277,  0.03488091,  0.02627761, ...,  0.01857099,
         -0.0156446 , -0.00622499],
        [-0.01239185,  0.00825603,  0.01180784, ...,  0.01308419,
         -0.02513637, -0.01149639],
        [-0.00300455, -0.00262631,  0.02177223, ...,  0.01372966,
         -0.03213968,  0.00140538],
        [-0.00210677, -0.00996671,  0.00578952, ..., -0.00493928,
          0.00881039,  0.00110918],
        [-0.00770416, -0.03298219,  0.00641239, ...,  0.0049683 ,
         -0.01183177,  0.01347619]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m

In [18]:

# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=1
)

[Document(id='5773d233-319c-43d5-bb57-b9a0e0d2eb18', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [19]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='5773d233-319c-43d5-bb57-b9a0e0d2eb18', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.41753751039505005),
 (Document(id='7582f4d6-fac4-49f1-9379-13b0b5aa51be', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.43137550354003906)]

In [20]:

# meta-data filtering
vector_store.similarity_search_with_score(
    query=" ",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='7582f4d6-fac4-49f1-9379-13b0b5aa51be', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.4798821210861206),
 (Document(id='029250b9-2684-4522-b414-94a3adb95a88', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.5008953213691711)]

In [21]:

# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

In [22]:
vector_store.update_document(document_id='f7bbffc0-e2a7-4876-9509-facf24a6c91e',document=updated_doc1)

In [23]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['8e8317a5-8519-4222-97e3-10b7a1152b40',
  'f9fbe472-3b41-4c1a-a73c-bbbdab36571e',
  '029250b9-2684-4522-b414-94a3adb95a88',
  '5773d233-319c-43d5-bb57-b9a0e0d2eb18',
  '7582f4d6-fac4-49f1-9379-13b0b5aa51be'],
 'embeddings': array([[ 0.00187277,  0.03488091,  0.02627761, ...,  0.01857099,
         -0.0156446 , -0.00622499],
        [-0.01239185,  0.00825603,  0.01180784, ...,  0.01308419,
         -0.02513637, -0.01149639],
        [-0.00300455, -0.00262631,  0.02177223, ...,  0.01372966,
         -0.03213968,  0.00140538],
        [-0.00210677, -0.00996671,  0.00578952, ..., -0.00493928,
          0.00881039,  0.00110918],
        [-0.00770416, -0.03298219,  0.00641239, ...,  0.0049683 ,
         -0.01183177,  0.01347619]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m

In [24]:

vector_store.delete(ids=['f7bbffc0-e2a7-4876-9509-facf24a6c91e',
  '903a37e2-fe77-47cb-b6a1-6604e92d8cdc'])

In [25]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['8e8317a5-8519-4222-97e3-10b7a1152b40',
  'f9fbe472-3b41-4c1a-a73c-bbbdab36571e',
  '029250b9-2684-4522-b414-94a3adb95a88',
  '5773d233-319c-43d5-bb57-b9a0e0d2eb18',
  '7582f4d6-fac4-49f1-9379-13b0b5aa51be'],
 'embeddings': array([[ 0.00187277,  0.03488091,  0.02627761, ...,  0.01857099,
         -0.0156446 , -0.00622499],
        [-0.01239185,  0.00825603,  0.01180784, ...,  0.01308419,
         -0.02513637, -0.01149639],
        [-0.00300455, -0.00262631,  0.02177223, ...,  0.01372966,
         -0.03213968,  0.00140538],
        [-0.00210677, -0.00996671,  0.00578952, ..., -0.00493928,
          0.00881039,  0.00110918],
        [-0.00770416, -0.03298219,  0.00641239, ...,  0.0049683 ,
         -0.01183177,  0.01347619]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m